# 04 — Train the 2D CNN
Vellakkal Dumpsite methane hotspot project.

Trains `build_custom_cnn()` on the leakage-safe train/val/test splits (Batch 4), using train-only normalization statistics and class-weighted binary cross-entropy. Saves the model plus its exact preprocessing/label configuration for reproducible inference later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/methane-hotspot-detection'
import os
os.makedirs(f'{PROJECT_ROOT}/models', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/outputs/figures', exist_ok=True)

In [ ]:
import sys
sys.path.append(f'{PROJECT_ROOT}/src')

!python {PROJECT_ROOT}/src/model.py  # prints the architecture summary as a sanity check

## Run training
Requires `patches_array.npz` (Batch 3) and the split files (Batch 4) to already exist.

In [ ]:
!cd {PROJECT_ROOT} && python src/train.py \
    --patches data/processed/patches_array.npz \
    --splits-dir data/splits \
    --models-dir models \
    --epochs 150 \
    --batch-size 32

## Inspect training curves and saved run


In [ ]:
import glob, json
run_dirs = sorted(glob.glob(f'{PROJECT_ROOT}/models/cnn_v1_*'))
latest_run = run_dirs[-1]
print('Latest run:', latest_run)

from PIL import Image
display(Image.open(f'{PROJECT_ROOT}/outputs/figures/training_curves.png'))

with open(f'{latest_run}/preprocessing_config.json') as f:
    print(json.dumps(json.load(f), indent=2))
with open(f'{latest_run}/label_threshold.json') as f:
    print(json.dumps(json.load(f), indent=2))

## Quick check: did validation AUC actually improve over training?
A fast red-flag check before moving to full evaluation (Batch 6) — confirms the run is worth evaluating and not a degenerate fit.

In [ ]:
import pandas as pd
hist = pd.read_csv(f'{latest_run}/training_history.csv')
print(hist[['auc', 'val_auc', 'loss', 'val_loss']].tail(10))
print(f"\nBest val_auc reached: {hist['val_auc'].max():.4f} at epoch {hist['val_auc'].idxmax()+1}")

Next: Batch 6 (`evaluate.py`) runs full test-set evaluation — confusion matrix, ROC/PR curves, and the honest comparison against the 1D CNN tabular baseline.